In [1]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.mamba import Mamba, MambaConfig

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
class BiMambaWrapper(nn.Module):
    def __init__(self, d_model, n_layers=2, d_state=16):
        super().__init__()
        config = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state)
        self.mamba_fwd = Mamba(config)
        self.mamba_bwd = Mamba(config)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        y_fwd = self.mamba_fwd(x)
        y_bwd = torch.flip(self.mamba_bwd(torch.flip(x, dims=[1])), dims=[1])
        return self.norm(y_fwd + y_bwd)


class ROIPatchEmbed(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64):
        super().__init__()
        self.grid_size = roi_size // patch_size
        self.n_tokens = (self.grid_size ** 3) * n_rois
        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.randn(1, self.n_tokens, d_model) * 0.02)

    def forward(self, rois):
        B, N = rois.shape[0], rois.shape[1]
        toks = [self.patch_conv(rois[:, i]).flatten(2).transpose(1, 2) for i in range(N)]
        return torch.cat(toks, dim=1) + self.pos_embed


class VisionMambaBranch(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64, n_layers=2, d_state=16):
        super().__init__()
        self.patch_embed = ROIPatchEmbed(n_rois, roi_size, patch_size, d_model)
        self.bimamba = BiMambaWrapper(d_model, n_layers, d_state)

    def forward(self, rois):
        tokens = self.patch_embed(rois)
        tokens = self.bimamba(tokens)
        return tokens.mean(dim=1)


class VisionMambaModel(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, rois):
        pooled = self.branch(rois)
        return self.classifier(self.dropout(pooled))


class MultimodalVisionMambaModel(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.mri_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.pet_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2, n_classes)

    def forward(self, mri_rois, pet_rois):
        fused = torch.cat([self.mri_branch(mri_rois), self.pet_branch(pet_rois)], dim=1)
        return self.classifier(self.dropout(fused))

In [3]:
COHORT_CSV    = "D:/mamba_model/thesis_cohort_final.csv"
MRI_CACHE_AUG = "D:/mamba_model/preprocessed_cache_roi64_aug"
PET_CACHE_AUG = "D:/mamba_model/preprocessed_cache_pet_aug"
CKPT_DIR      = "D:/mamba_model/checkpoints"

df = pd.read_csv(COHORT_CSV)
sessions = df["mri_session"].values
labels   = df["outcome_label"].values

X_tv, X_test, y_tv, y_test = train_test_split(
    sessions, labels, test_size=0.2, random_state=42, stratify=labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv
)
session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))

print(f"Val: {len(X_val)} | Test: {len(X_test)}")

Val: 42 | Test: 42


In [4]:
class ROIDataset(Dataset):
    def __init__(self, sessions, labels, cache_dir, is_mri=True, is_train=False):
        self.samples = []
        self.cache_dir = cache_dir
        for session_id, label in zip(sessions, labels):
            key = session_id if is_mri else session_to_subject[session_id]
            self.samples.append((key, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((key, label, f"aug{seed}"))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        key, label, version = self.samples[idx]
        rois = np.load(f"{self.cache_dir}/{key}_{version}.npy").astype(np.float32)
        return torch.tensor(rois).unsqueeze(1), torch.tensor(label, dtype=torch.long)


class MultimodalROIDataset(Dataset):
    def __init__(self, sessions, labels, mri_cache_dir, pet_cache_dir, is_train=False):
        self.samples = []
        self.mri_cache_dir = mri_cache_dir
        self.pet_cache_dir = pet_cache_dir
        for session_id, label in zip(sessions, labels):
            subject_id = session_to_subject[session_id]
            self.samples.append((session_id, subject_id, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((session_id, subject_id, label, f"aug{seed}"))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        mri_key, pet_key, label, version = self.samples[idx]
        mri_rois = np.load(f"{self.mri_cache_dir}/{mri_key}_{version}.npy").astype(np.float32)
        pet_rois = np.load(f"{self.pet_cache_dir}/{pet_key}_{version}.npy").astype(np.float32)
        return (torch.tensor(mri_rois).unsqueeze(1), torch.tensor(pet_rois).unsqueeze(1),
                torch.tensor(label, dtype=torch.long))

In [5]:
BATCH_SIZE = 4

mri_val_loader  = DataLoader(ROIDataset(X_val, y_val, MRI_CACHE_AUG, is_mri=True, is_train=False),
                              batch_size=BATCH_SIZE, shuffle=False)
mri_test_loader = DataLoader(ROIDataset(X_test, y_test, MRI_CACHE_AUG, is_mri=True, is_train=False),
                              batch_size=BATCH_SIZE, shuffle=False)

pet_val_loader  = DataLoader(ROIDataset(X_val, y_val, PET_CACHE_AUG, is_mri=False, is_train=False),
                              batch_size=BATCH_SIZE, shuffle=False)
pet_test_loader = DataLoader(ROIDataset(X_test, y_test, PET_CACHE_AUG, is_mri=False, is_train=False),
                              batch_size=BATCH_SIZE, shuffle=False)

mm_val_loader  = DataLoader(MultimodalROIDataset(X_val, y_val, MRI_CACHE_AUG, PET_CACHE_AUG, is_train=False),
                             batch_size=BATCH_SIZE, shuffle=False)
mm_test_loader = DataLoader(MultimodalROIDataset(X_test, y_test, MRI_CACHE_AUG, PET_CACHE_AUG, is_train=False),
                             batch_size=BATCH_SIZE, shuffle=False)

In [6]:
def find_best_threshold(model, val_loader, device, is_multimodal=False):
    """Sweeps decision thresholds on validation data to maximize balanced
    accuracy ((TPR+TNR)/2), instead of the default argmax (0.5 threshold)."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            if is_multimodal:
                mri_rois, pet_rois, lbls = batch
                mri_rois, pet_rois = mri_rois.to(device), pet_rois.to(device)
                outputs = model(mri_rois, pet_rois)
            else:
                rois, lbls = batch
                rois = rois.to(device)
                outputs = model(rois)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(lbls.numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    best_thresh, best_balanced_acc = 0.5, 0
    for thresh in np.arange(0.05, 0.96, 0.01):
        preds = (all_probs >= thresh).astype(int)
        tpr = recall_score(all_labels, preds, zero_division=0)
        tnr = specificity_score(all_labels, preds)
        balanced_acc = (tpr + tnr) / 2
        if balanced_acc > best_balanced_acc:
            best_balanced_acc = balanced_acc
            best_thresh = thresh

    return best_thresh, best_balanced_acc


def evaluate_with_threshold(model, loader, device, threshold, is_multimodal=False):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            if is_multimodal:
                mri_rois, pet_rois, lbls = batch
                mri_rois, pet_rois = mri_rois.to(device), pet_rois.to(device)
                outputs = model(mri_rois, pet_rois)
            else:
                rois, lbls = batch
                rois = rois.to(device)
                outputs = model(rois)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(lbls.numpy())

    preds = (np.array(all_probs) >= threshold).astype(int)
    all_labels = np.array(all_labels)
    acc = np.mean(preds == all_labels)
    tpr = recall_score(all_labels, preds, zero_division=0)
    tnr = specificity_score(all_labels, preds)
    return acc, tpr, tnr

In [7]:
mri_model = VisionMambaModel(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
mri_model.load_state_dict(torch.load(f"{CKPT_DIR}/vim_mri_only_d32_seed42.pt", weights_only=True))

best_thresh_mri, val_balanced_acc = find_best_threshold(mri_model, mri_val_loader, device)
acc, tpr, tnr = evaluate_with_threshold(mri_model, mri_test_loader, device, best_thresh_mri)

print(f"MRI-only -- calibrated threshold: {best_thresh_mri:.2f} (val balanced acc: {val_balanced_acc*100:.1f}%)")
print(f"MRI-only TEST at calibrated threshold: Acc={acc*100:.1f}% | TPR={tpr*100:.1f}% | TNR={tnr*100:.1f}%")
print(f"(For comparison, default argmax/0.5 threshold gave: Acc=71.4% | TPR=85.7% | TNR=57.1%)")

MRI-only -- calibrated threshold: 0.62 (val balanced acc: 71.4%)
MRI-only TEST at calibrated threshold: Acc=66.7% | TPR=61.9% | TNR=71.4%
(For comparison, default argmax/0.5 threshold gave: Acc=71.4% | TPR=85.7% | TNR=57.1%)


In [8]:
pet_model = VisionMambaModel(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
pet_model.load_state_dict(torch.load(f"{CKPT_DIR}/vim_pet_only_d32_seed42.pt", weights_only=True))

best_thresh_pet, val_balanced_acc = find_best_threshold(pet_model, pet_val_loader, device)
acc, tpr, tnr = evaluate_with_threshold(pet_model, pet_test_loader, device, best_thresh_pet)

print(f"PET-only -- calibrated threshold: {best_thresh_pet:.2f} (val balanced acc: {val_balanced_acc*100:.1f}%)")
print(f"PET-only TEST at calibrated threshold: Acc={acc*100:.1f}% | TPR={tpr*100:.1f}% | TNR={tnr*100:.1f}%")
print(f"(For comparison, default argmax/0.5 threshold gave: Acc=69.0% | TPR=76.2% | TNR=61.9%)")

PET-only -- calibrated threshold: 0.47 (val balanced acc: 76.2%)
PET-only TEST at calibrated threshold: Acc=69.0% | TPR=76.2% | TNR=61.9%
(For comparison, default argmax/0.5 threshold gave: Acc=69.0% | TPR=76.2% | TNR=61.9%)


In [9]:
mm_model = MultimodalVisionMambaModel(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
mm_model.load_state_dict(torch.load(f"{CKPT_DIR}/vim_multimodal_d32_seed42.pt", weights_only=True))

best_thresh_mm, val_balanced_acc = find_best_threshold(mm_model, mm_val_loader, device, is_multimodal=True)
acc, tpr, tnr = evaluate_with_threshold(mm_model, mm_test_loader, device, best_thresh_mm, is_multimodal=True)

print(f"Multimodal -- calibrated threshold: {best_thresh_mm:.2f} (val balanced acc: {val_balanced_acc*100:.1f}%)")
print(f"Multimodal TEST at calibrated threshold: Acc={acc*100:.1f}% | TPR={tpr*100:.1f}% | TNR={tnr*100:.1f}%")

Multimodal -- calibrated threshold: 0.63 (val balanced acc: 71.4%)
Multimodal TEST at calibrated threshold: Acc=64.3% | TPR=47.6% | TNR=81.0%
